In [ ]:
from pathlib import Path
import pandas as pd

# Assuming this notebook is in .../handwriting_forge_project/notebooks/
PROJECT_ROOT = Path.cwd().parent
META_DIR = PROJECT_ROOT / "data" / "processed" / "metadata"

DOMAIN_CLF_META_PATH = META_DIR / "domain_classification_sentences.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("DOMAIN_CLF_META_PATH:", DOMAIN_CLF_META_PATH)

# Load combined IAM+Emuru dataset
domain_clf_df = pd.read_csv(DOMAIN_CLF_META_PATH)

print("\nDomain classification dataset shape:", domain_clf_df.shape)
print("Columns:", list(domain_clf_df.columns))

print("\nLabel value counts:")
print(domain_clf_df["label"].value_counts())

print("\nDomain_label value counts (0=IAM, 1=Emuru):")
print(domain_clf_df["domain_label"].value_counts())

print("\nHead:")
display(domain_clf_df.head(5))


In [ ]:
from PIL import Image
import torch
from torchvision.transforms import functional as F

# We'll normalize all images to this height
TARGET_HEIGHT = 64

def load_line_image(img_path: str | Path, target_height: int = TARGET_HEIGHT) -> torch.Tensor:
    """
    Load a line image (IAM or Emuru), convert to grayscale, and
    resize so:
      - height = target_height
      - width is scaled proportionally
    Returns a float tensor of shape (1, H, W) in [0, 1].
    """
    img_path = Path(img_path)
    img = Image.open(img_path).convert("L")  # grayscale

    w, h = img.size
    if h == 0:
        raise ValueError(f"Invalid image height 0 for {img_path}")

    # scale width to keep aspect ratio when setting height = target_height
    scale = target_height / h
    new_w = max(1, int(round(w * scale)))

    img = img.resize((new_w, target_height), Image.BILINEAR)

    # to tensor: shape (1, H, W), values in [0,1]
    t = F.to_tensor(img)
    return t


In [ ]:
import random
from IPython.display import display

def tensor_to_pil(t: torch.Tensor) -> Image.Image:
    """
    Convert a (1, H, W) tensor in [0,1] to a PIL grayscale image.
    """
    t = t.clone().detach()
    if t.ndim == 3 and t.shape[0] == 1:
        t = t.squeeze(0)  # (H, W)
    t = t.clamp(0, 1)
    arr = (t.numpy() * 255).astype("uint8")
    return Image.fromarray(arr, mode="L")

k = 2  

iam_rows = domain_clf_df[domain_clf_df["domain_label"] == 0].sample(k)  # random every time
emu_rows = domain_clf_df[domain_clf_df["domain_label"] == 1].sample(k)  # random every time

for name, rows in [("IAM (genuine)", iam_rows), ("Emuru (fake)", emu_rows)]:
    print("=" * 80)
    print(name, f"- showing {len(rows)} random samples")
    for _, row in rows.iterrows():
        print("-" * 40)
        print("filepath:", row["filepath"])
        print("label:", row["label"], "| domain_label:", row["domain_label"])
        print("text:", repr(row["text"])[:120])
        
        img_path = PROJECT_ROOT / row["filepath"]
        t = load_line_image(img_path)  # (1, H, W)
        print("tensor shape (C, H, W):", tuple(t.shape))
        
        pil_proc = tensor_to_pil(t)
        display(pil_proc)


In [16]:
from torch.utils.data import Dataset

class DomainClassificationDataset(Dataset):
    """
    Dataset for IAM vs Emuru domain classification.
    Uses domain_clf_df with columns:
      - filepath (relative to PROJECT_ROOT)
      - domain_label (0 = IAM, 1 = Emuru)
      - source, text, idx, hf_split
    """

    def __init__(self, df: pd.DataFrame, project_root: Path, split: str | None = None):
        """
        df         : the full domain_clf_df or a filtered version
        project_root : Path to project root (for resolving filepaths)
        split      : if given, filter rows where hf_split == split (e.g. "train", "test")
        """
        if split is not None:
            df = df[df["hf_split"] == split].copy().reset_index(drop=True)

        self.df = df.reset_index(drop=True)
        self.project_root = Path(project_root)

        print(f"Initialized DomainClassificationDataset with {len(self.df)} samples"
              + (f" (split={split})" if split is not None else ""))

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx: int):
        row = self.df.iloc[idx]

        img_path = self.project_root / row["filepath"]
        img_tensor = load_line_image(img_path)  # (1, H, W), H=TARGET_HEIGHT

        label = int(row["domain_label"])  # 0 or 1

        sample = {
            "image": img_tensor,
            "label": label,
            "source": row["source"],
            "idx": int(row["idx"]),
            "text": row["text"],
            "hf_split": row["hf_split"],
        }
        return sample


In [21]:
from torch.utils.data import DataLoader
import torch
import torch.nn.functional as F

def pad_collate_fn(batch):
    """
    Custom collate_fn for variable-width line images.

    batch: list of samples from DomainClassificationDataset, each like:
      {
        "image": (1, H, W),
        "label": 0/1,
        "source": ...,
        "idx": ...,
        "text": ...,
        "hf_split": ...
      }

    Returns a dict with:
      - images: (B, 1, H, W_max)
      - labels: (B,)
      - sources, idxs, texts, hf_splits: lists
    """
    # images & labels
    images = [item["image"] for item in batch]   # list of (1, H, W)
    labels = [item["label"] for item in batch]

    # All heights should be equal (H = TARGET_HEIGHT); widths can vary
    heights = [img.shape[1] for img in images]
    widths  = [img.shape[2] for img in images]

    H = heights[0]
    max_W = max(widths)

    # Pad each image on the right to width = max_W
    padded_images = []
    for img in images:
        _, h, w = img.shape
        pad_right = max_W - w
        # pad = (left, right, top, bottom)
        padded = F.pad(img, (0, pad_right, 0, 0), value=0.0)
        padded_images.append(padded)

    batch_images = torch.stack(padded_images, dim=0)  # (B, 1, H, max_W)
    batch_labels = torch.tensor(labels, dtype=torch.long)

    # Metadata as lists
    sources   = [item["source"]   for item in batch]
    idxs      = [item["idx"]      for item in batch]
    texts     = [item["text"]     for item in batch]
    hf_splits = [item["hf_split"] for item in batch]

    return {
        "images": batch_images,
        "labels": batch_labels,
        "sources": sources,
        "idxs": idxs,
        "texts": texts,
        "hf_splits": hf_splits,
    }

# --- Test: make a small dataset + loader and inspect one batch ---

# Here we use the whole dataframe; later you can filter by split (train/test)
test_ds = DomainClassificationDataset(domain_clf_df, PROJECT_ROOT)

test_loader = DataLoader(
    test_ds,
    batch_size=4,
    shuffle=True,
    collate_fn=pad_collate_fn,
)

# Get one batch
batch = next(iter(test_loader))

print("Batch images shape:", batch["images"].shape)  # (B, 1, H, W_max)
print("Batch labels:", batch["labels"])
print("Sources:", batch["sources"])
print("Idxs:", batch["idxs"])


Initialized DomainClassificationDataset with 19602 samples
Batch images shape: torch.Size([4, 1, 64, 1410])
Batch labels: tensor([1, 0, 0, 1])
Sources: ['emuru', 'iam', 'iam', 'emuru']
Idxs: [6269, 6940, 3918, 1661]


In [22]:
from torch.utils.data import DataLoader

# What splits do we have?
print("Unique hf_split values:", domain_clf_df["hf_split"].unique())

splits = set(domain_clf_df["hf_split"].unique())

train_ds = None
val_ds = None
test_ds = None

# Create datasets only for splits that exist
if "train" in splits:
    train_ds = DomainClassificationDataset(domain_clf_df, PROJECT_ROOT, split="train")
if "validation" in splits:
    val_ds = DomainClassificationDataset(domain_clf_df, PROJECT_ROOT, split="validation")
if "valid" in splits:
    val_ds = DomainClassificationDataset(domain_clf_df, PROJECT_ROOT, split="valid")
if "test" in splits:
    test_ds = DomainClassificationDataset(domain_clf_df, PROJECT_ROOT, split="test")

# Show what we got
print("\nDatasets:")
print("  train_ds:", len(train_ds) if train_ds is not None else None)
print("  val_ds:  ", len(val_ds)   if val_ds   is not None else None)
print("  test_ds: ", len(test_ds)  if test_ds  is not None else None)

# Create DataLoaders (only for non-None datasets)
batch_size = 32

train_loader = None
val_loader = None
test_loader = None

if train_ds is not None:
    train_loader = DataLoader(
        train_ds,
        batch_size=batch_size,
        shuffle=True,
        collate_fn=pad_collate_fn,
    )

if val_ds is not None:
    val_loader = DataLoader(
        val_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=pad_collate_fn,
    )

if test_ds is not None:
    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        collate_fn=pad_collate_fn,
    )

# Inspect one batch from train_loader (if it exists)
if train_loader is not None:
    batch = next(iter(train_loader))
    print("\nTrain batch images shape:", batch["images"].shape)  # (B, 1, 64, W_max)
    print("Train batch labels:", batch["labels"][:10])
    print("Train batch sources:", batch["sources"][:10])
else:
    print("\nNo 'train' split found – you might only have one split (e.g. 'train').")


Unique hf_split values: ['test' 'train' 'validation']
Initialized DomainClassificationDataset with 12256 samples (split=train)
Initialized DomainClassificationDataset with 1840 samples (split=validation)
Initialized DomainClassificationDataset with 5506 samples (split=test)

Datasets:
  train_ds: 12256
  val_ds:   1840
  test_ds:  5506

Train batch images shape: torch.Size([32, 1, 64, 2236])
Train batch labels: tensor([0, 0, 0, 0, 1, 1, 0, 1, 1, 1])
Train batch sources: ['iam', 'iam', 'iam', 'iam', 'emuru', 'emuru', 'iam', 'emuru', 'emuru', 'emuru']


In [19]:
from torchvision.transforms import functional as TF  # new alias

TARGET_HEIGHT = 64  # keep the same target height

def load_line_image(img_path: str | Path, target_height: int = TARGET_HEIGHT) -> torch.Tensor:
    """
    Load a line image (IAM or Emuru), convert to grayscale, and
    resize so:
      - height = target_height
      - width is scaled proportionally
    Returns a float tensor of shape (1, H, W) in [0, 1].
    """
    img_path = Path(img_path)
    img = Image.open(img_path).convert("L")  # grayscale

    w, h = img.size
    if h == 0:
        raise ValueError(f"Invalid image height 0 for {img_path}")

    # scale width to keep aspect ratio when setting height = target_height
    scale = target_height / h
    new_w = max(1, int(round(w * scale)))

    img = img.resize((new_w, target_height), Image.BILINEAR)

    # to tensor: shape (1, H, W), values in [0,1]
    t = TF.to_tensor(img)
    return t


In [23]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class DomainCNN(nn.Module):
    """
    Simple CNN for IAM vs Emuru domain classification.
    Input:  (B, 1, 64, W)
    Output: (B, 2) logits (class 0 = genuine/IAM, class 1 = fake/Emuru)
    """

    def __init__(self):
        super().__init__()
        # Feature extractor
        self.conv_block = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),   # (64 -> 32) in height, W/2 in width

            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),   # (32 -> 16)

            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            # no pool here; we'll do global pooling later
        )

        # Classification head
        self.fc = nn.Linear(128, 2)  # 2 classes: genuine vs fake

    def forward(self, x):
        """
        x: (B, 1, 64, W)
        """
        feat = self.conv_block(x)  # (B, 128, H', W')

        # Global average pooling over (H', W')
        feat = feat.mean(dim=[2, 3])  # (B, 128)

        logits = self.fc(feat)        # (B, 2)
        return logits

# Choose device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Instantiate model
model = DomainCNN().to(device)
print(model)


Using device: cpu
DomainCNN(
  (conv_block): Sequential(
    (0): Conv2d(1, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): ReLU(inplace=True)
    (3): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (4): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (5): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (6): ReLU(inplace=True)
    (7): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (8): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (9): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (10): ReLU(inplace=True)
  )
  (fc): Linear(in_features=128, out_features=2, bias=True)
)
